In [1]:
from dotenv import load_dotenv
load_dotenv()
import os


In [2]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [3]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

In [ ]:

model="deepseek-r1-distill-llama-70b"
llm=ChatGroq(model=model)
llm.invoke('hi how r u')

In [5]:
#llm = ChatOpenAI()
# llm.invoke("Hi")

In [6]:
#in Open AI we no need to mention the mode name
# llm=ChatOpenAI()
# ll.invoke("hi how r u")
# from langchain_openai import OpenAIEmbeddings

# embedding=OpenAIEmbeddings(
#     model="text-embedding-3-large"
# )
# embedding.embed_query("hi")

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")

embeddings.embed_query("hi how r u")


In [ ]:
#will take a document from the web browser
from langchain_community.document_loaders import WebBaseLoader

url="https://lilianweng.github.io/posts/2023-06-23-agent/"
web_loader=WebBaseLoader(url)

data=web_loader.load()

In [ ]:
from rich import print as rprint
from rich.pretty import pretty_repr
rprint("data -> ",data[0].metadata)
print(data[0].metadata["description"])

len(data[0].metadata["description"])

In [10]:
urls=[
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
]

docs=[WebBaseLoader(url).load() for url in urls]

In [ ]:
#docs
#since the doc is in list inside list, so we uses sublist to make it a single list
doc_list=[item for sublist in docs for item in sublist]
print(doc_list)

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=100,chunk_overlap=25)


doc_splits=text_splitter.split_documents(doc_list)

In [ ]:
doc_splits

In [ ]:


import numpy as np
from langchain_community.vectorstores import Chroma
doc_embeddings = [embeddings.embed_query(doc.page_content) for doc in doc_splits]
# Convert to numpy array for similarity calculations
stored_embeddings = np.array(doc_embeddings)

stored_embeddings
db=Chroma.from_documents(doc_splits,embeddings)
retriver=db.as_retriever(search_kwargs={"k":3})
msg="Industry"
retriver.invoke(msg)

In [15]:
from langchain.tools.retriever import create_retriever_tool
# convertivng retiver into a tool langchain has that facility


In [16]:
retriever_tool = create_retriever_tool(
    retriver,  # your retriever object
    "retriever_blog_post",  # tool name
    description="Use this tool to search blog posts and retrieve relevant content."  # required
)

In [17]:
tools=[retriever_tool]
# to built it wiht the graph tool
from langgraph.prebuilt import ToolNode

retriver_node=ToolNode(tools)

In [ ]:
llm_with_tool=llm.bind_tools(tools)

response=llm_with_tool.invoke("what is India")

response.tool_calls


LANGGRAPH ORCHESTRATION

In [19]:
import operator
from pydantic import BaseModel,Field
from typing import TypedDict, Annotated, Sequence
from langchain.prompts import ChatPromptTemplate,PromptTemplate

from langchain_core.messages import BaseMessage
from langchain_core.messages import HumanMessage,AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain import hub
#we need to mention litterals for type hinting

from typing import Literal



In [20]:
class Agentstate(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

In [21]:
class grade(BaseModel):
    binary_score:str=Field(description="relevance score 'yes' or 'no' ")

In [22]:
# message=state["messages"]

In [23]:
# def LLM_Decision_Maker(state:Agentstate):
#     print("----CALL LLM_DECISION_MAKE----")
#     message=state["messages"]
    
#     if len(message)>1:
#         last_message=message[-1]
#         question=last_message.content
        
#         prompt=PromptTemplate(
#         template="""You are a helpful assistant whatever question has been asked to find out that in the given question and answer.
#                         Here is the question:{question}
#                         """,
#                         input_variables=["question"]
#                         )
#         chain=prompt | llm
#         response=chain.invoke({"question":question})
#         return {"messages":[response]}
#     else:
#         llm_with_tool=llm.bind_tools(tools)
#         response=llm_with_tool.invoke(message)
#         return {"messages":[response]}

In [24]:

def LLM_Decision_Maker(state:Agentstate):
    print("----CALL LLM_DECISION_MAKE----")
    message=state["messages"]
    print(message)
    last_message=message[-1]
    print("question : ",last_message.content)
    question=state["messages"][-1].content
    response=llm_with_tool.invoke(question)
    print("response :",response)
    return {"messages":[response]}

In [ ]:
from langchain.schema import HumanMessage, AIMessage
state = {
    "messages": [HumanMessage(content="USA trade data")]
}
LLM_Decision_Maker(state)

In [ ]:
rprint(llm)
rprint(tools)

In [ ]:
llm_with_structure_op=llm.with_structured_output(grade)
rprint(llm_with_structure_op)

In [28]:
def grade_documents(state:Agentstate)->Literal["Output Generator", "Query Rewriter"]:
    print("----CALLING GRADE FOR CHECKING RELEVANCY----")
    llm_with_structure_op=llm.with_structured_output(grade)
    
    prompt=PromptTemplate(
        template="""You are a grader deciding if a document is relevant to a user’s question.
                    Here is the document: {context}
                    Here is the user’s question: {question}
                    If the document talks about or contains information related to the user’s question, mark it as relevant. 
                    Give a 'yes' or 'no' answer to show if the document is relevant to the question.""",
                    input_variables=["context", "question"]
                    )
     
    chain=prompt|llm_with_structure_op
     
     
    message=state['messages']
    
    last_message = message[-1]
    
    question = message[0].content
    
    docs = last_message.content
    
    scored_result=chain.invoke({"question": question, "context": docs})
    
    score=scored_result.binary_score
     
    if score=="yes":
        print("----DECISION: DOCS ARE RELEVANT----")
        return "generator"
    else:
        print("----DECISION: DOCS ARE NOT RELEVANT----")
        return "rewrite"

In [ ]:
prompt = hub.pull("rlm/rag-prompt")  # Correct spelling
print(prompt)

In [30]:
def generate(state:Agentstate):
    print("----RAG OUTPUT GENERATE----")
    
    message=state["messages"]
    question=message[0].content
    
    last_message = message[-1]
    docs = last_message.content
    
    prompt=hub.pull("rlm/rag-prompt")
    
    rag_chain=prompt | llm
    
    response=rag_chain.invoke({"context": docs, "question": question})
    
    print(f"this is my response:{response}")
    
    return {"messages": [response]}

In [31]:
def rewrite(state:Agentstate):
    print("----TRANSFORM QUERY----")
    message=state["messages"]
    
    question=message[0].content
    
    input= [HumanMessage(content=f"""Look at the input and try to reason about the underlying semantic intent or meaning. 
                    Here is the initial question: {question} 
                    Formulate an improved question: """)
       ]

    response=llm.invoke(input)
    
    return {"messages": [response]}

In [ ]:
rewrite(state)

In [33]:
from langgraph.graph import StateGraph,END,START

In [ ]:
workflow=StateGraph(Agentstate)

workflow.add_node("LLM_Decision_Maker",LLM_Decision_Maker)
workflow.add_node("vector Retriver",retriver_node)
workflow.add_node("output Generator",generate)
workflow.add_node("Query Rewriter",rewrite)


In [ ]:
workflow.add_edge(START,"LLM_Decision_Maker")

from langgraph.prebuilt import tools_condition
workflow.add_conditional_edges("LLM_Decision_Maker",
                               tools_condition,
                               {
                                "tools":"vector Retriver",
                                END:END
                               }
                               )


workflow.add_conditional_edges("vector Retriver",
                               grade_documents,
                               {
                                "generator":"output Generator",
                                "rewrite":"Query Rewriter"
                               
                               }
                               )

In [ ]:
workflow.add_edge("output Generator",END)

workflow.add_edge("Query Rewriter","LLM_Decision_Maker")

In [ ]:
workflow.compile()

In [38]:
app=workflow.compile()

In [39]:
import warnings
warnings.filterwarnings("ignore")

In [40]:
#app.invoke({"messages":[HumanMessage("what is LLM Powered Autonomous Agents explain the planning and reflection and prompt engineering explain me in terms of agents and langchain?")]})

In [ ]:
result=app.invoke({"messages":[HumanMessage("Could you provide an update on the latest stock performance and news related to Futu Holdings (FUTU)?")]})

In [ ]:
result=app.invoke({"messages":[HumanMessage("capital of USA")]})


In [ ]:
rprint(result["messages"][-1].content)